In [ ]:
!pip install -q transformers torch faiss-cpu Pillow tqdm h5py

In [ ]:
import os, time, numpy as np, torch, torch.nn.functional as F, tarfile
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoImageProcessor
import faiss, h5py, pickle

# Universal secret getter
def get_secret(key: str, fallback_path: str = None, default=None) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key)
    except Exception:
        pass
    if fallback_path and os.path.exists(fallback_path):
        return open(fallback_path).read().strip()
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    value = os.environ.get(key, default)
    if value is None:
        raise EnvironmentError(f"Secret '{key}' not found")
    return value

hf_token = get_secret("HF_TOKEN", fallback_path="/kaggle/input/my-hf-secrets/HF_TOKEN.txt")
print("✓ HF_TOKEN loaded")

INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp"}

def find_and_extract_dataset():
    """Find dataset, extract tar if needed, return path with images"""
    if not INPUT_DIR.exists():
        return None
    for dataset_dir in sorted(INPUT_DIR.iterdir()):
        if not dataset_dir.is_dir():
            continue
        # Check for tar files first
        tar_files = list(dataset_dir.glob("*.tar")) + list(dataset_dir.glob("*.tar.gz"))
        if tar_files:
            extract_dir = WORK_DIR / "dataset"
            extract_dir.mkdir(exist_ok=True)
            print(f"Extracting {tar_files[0].name}...")
            with tarfile.open(tar_files[0], 'r:*') as tar:
                tar.extractall(extract_dir)
            print(f"✓ Extracted to {extract_dir}")
            return str(extract_dir)
        # Check if directory has images directly
        for ext in SUPPORTED_EXTS:
            if list(dataset_dir.rglob(f"*{ext}")):
                return str(dataset_dir)
    return None

DATASET_PATH = find_and_extract_dataset()
if not DATASET_PATH:
    raise FileNotFoundError(f"No dataset found in {INPUT_DIR}")
print(f"✓ Dataset: {DATASET_PATH}")

MODEL_ID = "facebook/dinov3-vitb16-pretrain-lvd1689m"
EMBEDDING_DIM = 768
OUTPUT_DIR = "/kaggle/working"

# Adaptive GPU/CPU configuration
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0)
    
    if "P100" in gpu_name:
        device, BATCH_SIZE, NUM_WORKERS = "cpu", 32, 2
        print(f"GPU: {gpu_name} (not supported) - using CPU")
    elif num_gpus > 1:
        device, BATCH_SIZE, NUM_WORKERS = "cuda", 320, 4
        print(f"GPU: {gpu_name} x{num_gpus} | Batch: {BATCH_SIZE}")
    else:
        device, BATCH_SIZE, NUM_WORKERS = "cuda", 256, 4
        print(f"GPU: {gpu_name} | Batch: {BATCH_SIZE}")
else:
    device, BATCH_SIZE, NUM_WORKERS = "cpu", 32, 2
    print("CPU mode")


In [ ]:
# Dataset loader
class FastDataset(Dataset):
    def __init__(self, paths, proc):
        self.paths, self.proc = paths, proc

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (224, 224), 0)

        inputs = self.proc(images=img, return_tensors="pt")
        inputs_squeezed = {k: v.squeeze(0) for k, v in inputs.items()}
        return inputs_squeezed, self.paths[i]


imgs = [
    (str(p.relative_to(DATASET_PATH)), str(p))
    for p in Path(DATASET_PATH).rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
]
image_ids, image_paths = [i for i, _ in imgs], [p for _, p in imgs]
print(f"Found {len(image_ids)} images")

In [ ]:
# Load model
print(f"Loading {MODEL_ID}...")
t0 = time.time()

processor = AutoImageProcessor.from_pretrained(MODEL_ID, token=hf_token)

dtype   = torch.float16 if device == "cuda" else torch.float32
use_amp = (device == "cuda")
if use_amp:
    print("  Using FP16 for speedup.")

model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    token=hf_token,
)

torch.cuda.empty_cache()

# Dense-feature wrapper
class DenseWrapper(torch.nn.Module):
    """Returns mean-pooled patch tokens (dense / texture embedding)."""

    def __init__(self, base_model):
        super().__init__()
        self.model = base_model

    def forward(self, **kwargs):
        outputs = self.model(**kwargs)
        patch_tokens = outputs.last_hidden_state[:, 1:, :]  # (B, P, D)
        dense_emb = patch_tokens.mean(dim=1)                # (B, D)
        return dense_emb


wrapped_model = DenseWrapper(model)

if device == "cuda" and torch.cuda.device_count() > 1:
    final_model = torch.nn.DataParallel(wrapped_model).cuda()
    print("  ✓ Applied DataParallel")
else:
    final_model = wrapped_model.to(device)

final_model.eval()
if device == "cuda":
    torch.backends.cudnn.benchmark = True

print(f"Model ready in {time.time() - t0:.1f}s")

In [ ]:
# Inference loop
loader = DataLoader(
    FastDataset(image_paths, processor),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(device == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
)

all_emb, t0 = [], time.time()

with torch.no_grad():
    for inputs, _ in tqdm(loader, desc="Dense inference"):
        inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}

        if use_amp:
            with torch.amp.autocast("cuda", dtype=torch.float16):
                dense_emb = final_model(**inputs)
                feat      = F.normalize(dense_emb, p=2, dim=-1)
        else:
            dense_emb = final_model(**inputs)
            feat      = F.normalize(dense_emb, p=2, dim=-1)

        all_emb.append(feat.cpu().float().numpy())

if device == "cuda":
    torch.cuda.synchronize()

inf_time   = time.time() - t0
embeddings = np.vstack(all_emb)
print(f"Done: {inf_time:.1f}s | {len(image_paths) / inf_time:.1f} img/s")
print(f"Embedding matrix shape: {embeddings.shape}")

In [ ]:
# Save HDF5 + FAISS index
hdf5_path  = OUTPUT_DIR + "/embeddings.hdf5"
faiss_path = OUTPUT_DIR + "/faiss_index.pkl"

with h5py.File(hdf5_path, "w") as f:
    f.create_dataset("embeddings",   data=embeddings)
    f.create_dataset("image_ids",    data=[s.encode("utf-8") for s in image_ids])
    f.create_dataset("image_paths",  data=[s.encode("utf-8") for s in image_paths])
    f.attrs["model_id"]       = MODEL_ID
    f.attrs["feature_type"]   = "dense_patch_mean_pool"
    f.attrs["embedding_dim"]  = EMBEDDING_DIM
    f.attrs["num_images"]     = len(image_ids)

print(f"✓ Saved HDF5  → {hdf5_path}")

index = faiss.IndexFlatIP(EMBEDDING_DIM)
index.add(embeddings.astype(np.float32))

with open(faiss_path, "wb") as f:
    pickle.dump(index, f)

print(f"✓ Saved FAISS → {faiss_path}")
print(f"  Total vectors indexed: {index.ntotal}")